In [1]:
import wandb
import pandas as pd
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

entity = "bsarec"
project = "new_datasets"

api = wandb.Api()
runs = api.runs(f"{entity}/{project}")

models = ["BERT4Rec", "FMLPRec", "duoRec", "FeaRec", "BSARec"]
models_seen = {name: False for name in models}
metrics = ["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]
columns = ["model_name", "dataset", "metric", "value"]
data = []

# only take the bsarec runs where c is 7,  lr is 0.005 attention heads is 1, alpha is 0.5 for the MIND dataset
# for taobao_small, c is 9, lr is 0.005, attention heads is 4, alpha is 0.7
# remove the other bsarec runs. Keep all the others

for run in runs:
    if run.state != "finished":
        continue

    model_name = run.config.get("model_type", None)

    if model_name is None or model_name not in models:
        continue


    models_seen[model_name] = True

    dataset = run.config.get("data_name", None)

    for metric_name in metrics:
        if run.config.get("model_type", None) == "BSARec":
            if run.config.get('data_name', None) == "MIND":
                if not (run.config.get("c", None) == 7 and
                        run.config.get("lr", None) == 0.005 and
                        run.config.get("num_attention_heads", None) == 1 and
                        run.config.get("alpha", None) == 0.5):
                    continue
            elif run.config.get('data_name', None) == "taobao_small":
                if not (run.config.get("c", None) == 9 and
                        run.config.get("lr", None) == 0.005 and
                        run.config.get("num_attention_heads", None) == 4 and
                        run.config.get("alpha", None) == 0.7):
                    continue
            # no not include run if it failed
            if run.summary.get("test/HR@5", None) is None:
                continue
        
        if run.config.get("epochs") < 200:
            continue
        metric_value = run.summary.get(f"test/{metric_name}", None)

        if metric_value is None:
            continue

        data.append([
            model_name,
            dataset,
            metric_name,
            metric_value
        ])

print("Models seen:", models_seen)


Models seen: {'BERT4Rec': True, 'FMLPRec': True, 'duoRec': True, 'FeaRec': False, 'BSARec': True}


In [5]:
df = pd.DataFrame(data, columns=columns)
df["dataset"] = df["dataset"].str.replace("_", " ")
df_immutable = df.copy(deep=True)




df.head(5)


,model_name,dataset,metric,value
0,BSARec,MIND,HR@5,0.114396
1,BSARec,MIND,HR@10,0.170947
2,BSARec,MIND,HR@20,0.243901
3,BSARec,MIND,NDCG@5,0.076420
4,BSARec,MIND,NDCG@10,0.094605


In [6]:
# num of entries per model
print("Number of entries per model:")
print(df["model_name"].value_counts())

Number of entries per model:
model_name
BSARec      90
BERT4Rec    42
FMLPRec     42
duoRec      18
Name: count, dtype: int64


# Paired T-test

In [7]:
df_ttest = df_immutable.copy(deep=True)
mean_df = df_ttest.groupby(['model_name', 'dataset', 'metric'])['value'].mean().reset_index()
pivot_df = mean_df.pivot_table(index=['dataset', 'metric'], columns='model_name', values='value')

results = []

DEBUG = False

target_model = "BSARec"
signficant_col = "significant"

for (dataset, metric), row in pivot_df.iterrows():
    # Drop target_model temporarily to find best *other* model
    row_wo_target = row.drop(labels=target_model, errors='ignore').sort_values(ascending=False)

    if row_wo_target.count() < 1:
        continue  # No valid models to compare against

    best_other_model = row_wo_target.index[0]

    # Extract values for both models
    target_vals = df_ttest[(df_ttest['model_name'] == target_model) &
                           (df_ttest['dataset'] == dataset) &
                           (df_ttest['metric'] == metric)]['value'].values

    best_vals = df_ttest[(df_ttest['model_name'] == best_other_model) &
                         (df_ttest['dataset'] == dataset) &
                         (df_ttest['metric'] == metric)]['value'].values

    # Ensure lengths match for paired t-test
    if len(target_vals) != len(best_vals):
        print(f"Skipping {dataset} - {metric}: Length mismatch between {target_model} and {best_other_model}")
        print(f"Lengths: {len(target_vals)} vs {len(best_vals)}")
        continue

    t_stat, p_value = ttest_rel(target_vals, best_vals)

    if DEBUG:
        results.append({
            'dataset': dataset,
            'metric': metric,
            'target_model': target_model,
            'best_other_model': best_other_model,
            'target_mean': target_vals.mean(),
            'best_other_mean': best_vals.mean(),
            signficant_col: p_value < 0.05,
        })
    else:
        results.append({
            'dataset': dataset,
            'metric': metric,
            signficant_col: p_value < 0.05,
        })

stats_df = pd.DataFrame(results)
stats_df = stats_df.set_index(["dataset", "metric"])
stats_df.head()


Skipping MIND - HR@10: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping MIND - HR@20: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping MIND - HR@5: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping MIND - NDCG@10: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping MIND - NDCG@20: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping MIND - NDCG@5: Length mismatch between BSARec and BERT4Rec
Lengths: 14 vs 4
Skipping taobao small - HR@10: Length mismatch between BSARec and BERT4Rec
Lengths: 1 vs 3
Skipping taobao small - HR@20: Length mismatch between BSARec and BERT4Rec
Lengths: 1 vs 3
Skipping taobao small - HR@5: Length mismatch between BSARec and BERT4Rec
Lengths: 1 vs 3
Skipping taobao small - NDCG@10: Length mismatch between BSARec and BERT4Rec
Lengths: 1 vs 3
Skipping taobao small - NDCG@20: Length mismatch between BSARec and BERT4Rec
Lengths: 1 vs 3
Skipping taobao small - NDCG@5: Le

KeyError: "None of ['dataset', 'metric'] are in the columns"

# Main Table

In [ ]:
df = df_immutable.copy(deep=True)
df = (
    df.groupby(["model_name", "dataset", "metric"])
      .agg(["mean"])
      .stack(future_stack=True)
      .reset_index()
      .pivot_table(
          index=['dataset', 'metric'],
          columns='model_name',
          values='value'
      )
)

def compute_percent_improvement(row, target_model="BSARec"):
    values = row.dropna().astype(float) # drop NaNs, convert to float
    target_model_value = values[target_model]
    values = values.drop(target_model)
    best_baseline = values.max()

    return ((target_model_value - best_baseline) / best_baseline) * 100

# create Improv. column with NaN for all rows first
rel_improvement_col= "Diff."
df[rel_improvement_col] = np.nan

# compute improvement only on 'mean' rows
df.loc[:, rel_improvement_col] = df.apply(compute_percent_improvement, axis=1)


In [ ]:
# add statistic significance column
df = df.join(stats_df, how="left")

ValueError: cannot join with no overlapping index names

In [ ]:
df.head()

In [ ]:
# sort metrics
df = df.copy().reset_index()
df["metric"] = pd.Categorical(df["metric"], categories=metrics, ordered=True)

df = df.sort_values(["dataset", "metric"])
df = df.set_index(["dataset", "metric"])

# re-order columns to match the models list
col_order = [mname for mname in models if mname in df.columns] + [rel_improvement_col, signficant_col]
df = df[col_order]

In [ ]:
def format_row(row: pd.Series, rel_improvement_col, significant_col) -> list:
    formatted_row = {}

    stat_cols = [rel_improvement_col, significant_col]

    row = pd.to_numeric(row, errors='coerce')
    top2 = row.drop(stat_cols).nlargest(2)
    first, second = top2.index.tolist()

    for name, val in row.items():
        if name in stat_cols:
            continue

        fval = val

        if isinstance(val, float):
            fval = f"{val:.4f}"
        elif isinstance(val, str):
            pass
        else:
            fval = str(val)

        if name == first:
            fval = "\\textbf{" + fval + "}"
        elif name == second:
            fval = "\\underline{" + fval + "}"

        formatted_row[name] = fval

    modifier = {
        True: "\\textsuperscript{*}",
        False: ""
    }

    formatted_row[rel_improvement_col] = f"{row[rel_improvement_col]:.2f}" + modifier.get(row[significant_col], " X")

    return pd.Series(formatted_row)

df = df.apply(lambda row: format_row(row, rel_improvement_col, signficant_col), axis=1)

In [ ]:


# create base latex string
# WARNING: hardcoded column count
latex = df.to_latex(escape=False, na_rep="", multicolumn=True, multirow=True)
latex = latex.replace("\\begin{tabular}{llllllll}", "\\begin{tabular}{llcccccc}\n\\toprule")
latex = latex.replace("\\\\\n", " \\\\\n\\midrule\n", 1)  # Add \midrule after header
latex = latex.replace("\\cline{1-9}\n\\bottomrule", "\\bottomrule")
latex = latex.replace("\\cline{1-9}", "\\midrule")

# fix the header line
lines = latex.splitlines()
toprule_idx = None
for i, line in enumerate(lines):
    if r"\toprule" in line:
        toprule_idx = i
        break

# Replace next two lines after \toprule with single header line
# Assuming they exist
header_line = r"\textbf{Dataset} & \textbf{Metric} &" + " & ".join([f"\\textbf{{{col}}}" for col in col_order if col != signficant_col]) + r" \\"
lines[toprule_idx + 1] = header_line

del lines[toprule_idx+2]

# insert \midrule after new header line if not already present
if r"\midrule" not in lines[toprule_idx+2]:
    lines.insert(toprule_idx+2, r"\midrule")

latex_fixed = "\n".join(lines)
latex_fixed = latex_fixed.replace("\\midrule\ndataset & metric &  &  &  &  &  &  &  \\\\\n", "")
latex_fixed = latex_fixed.replace("\\multirow[t]{6}{*}{", "\\multirow{6}{*}{\\centering ")
# latex_fixed = latex_fixed.replace("\\toprule", "", 1)
print(latex_fixed)